# Parity GPU run: Unsloth 2026.11.10

Step-by-step instructions are in `docs/gpu-run/GPU-RUN.md`. In short:

1. **Runtime > Change runtime type > T4 GPU > Save** (do this first).
2. Run each cell below in order with its ▶ button, and wait for each to finish.
   In cell 2, `RUN = "full"` runs the checks (about 25 minutes); `RUN = "p0005"` runs only P-0005 (a few minutes).
3. Cell 4 asks for a file: pick `parity-src-<commit>.zip` from the kit folder.
4. Cell 6 is optional: the P-0005 repro, added to the same zip in its own folder. Skipping it, or it failing, leaves
   the main results unchanged.
5. Cell 7 downloads `parity-gpu-results-....zip`. Hand that zip back.

No logins or tokens are needed or used.

In [ ]:
# Cell 2: which models to run. Each is "org/name@<40-character commit>".
# The committed default is the calibration model. `build_kit.py --models a,b` writes the list into the kit's copy,
# after checking that each target's predictions are committed. Each target also needs its predictions in the code
# zip: run_checks.py refuses a target without them.
MODELS = [
    "Qwen/Qwen3-0.6B@c1899de289a04d12100db370d81485cdf75e47ca",
]
# What to run:
#   "full"  - install, check the GPU route, then the six checks on MODELS (about 25 minutes). Cell 6 can add P-0005.
#   "p0005" - install, check the GPU route, then only P-0005 (its repro three ways and the adapter's pad_to check), with
#             no checks: a few minutes.
RUN = "full"

In [ ]:
# Cell 3: is there a GPU?
import shutil
import subprocess
if shutil.which("nvidia-smi") is None:
    raise RuntimeError("No GPU. Runtime > Change runtime type > T4 GPU > Save, then run the cells again from cell 2.")
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

In [ ]:
# Cell 4: upload the code (made by build_kit.py with git archive) and unpack it.
import hashlib
import pathlib
import re
import shutil
import zipfile
from google.colab import files

print("Click 'Choose files' and pick parity-src-<commit>.zip from the kit folder.")
uploaded = files.upload()
names = [n for n in uploaded if re.fullmatch(r"parity-src-[0-9a-f]{12}( \(\d+\))?\.zip", n)]
if len(names) != 1:
    raise RuntimeError(f"Upload exactly one parity-src-<commit>.zip (got {list(uploaded)}). Run this cell again.")
archive = pathlib.Path("/content") / names[0]
archive.write_bytes(uploaded[names[0]])
ARCHIVE_SHA256 = hashlib.sha256(archive.read_bytes()).hexdigest()
with zipfile.ZipFile(archive) as zf:
    COMMIT = zf.comment.decode("ascii", "replace")  # git archive stores the commit id here
    if not re.fullmatch(r"[0-9a-f]{40}", COMMIT) or not names[0].startswith(f"parity-src-{COMMIT[:12]}"):
        raise RuntimeError("This zip wasn't made by build_kit.py: its commit id is missing or doesn't match its name.")
    ROOT = pathlib.Path("/content/parity")
    if ROOT.exists():
        shutil.rmtree(ROOT)
    zf.extractall(ROOT)
print(f"Code from commit {COMMIT}")
print(f"Archive sha256  {ARCHIVE_SHA256}")

In [ ]:
# Cell 5: install Unsloth's pinned environment, check the GPU route, then run what cell 2's RUN says
# (RUN = "full": the checks, about 20 to 40 minutes; RUN = "p0005": only P-0005, a few minutes).
# Keep this browser tab open while it runs. Each model's results download as soon as that model finishes
# (parity-gpu-model-....zip); allow the downloads if the browser asks.
import subprocess
import sys
from google.colab import files
command = [
    sys.executable, str(ROOT / "docs" / "gpu-run" / "colab_kit.py"),
    "--models", ",".join(MODELS), "--commit", COMMIT, "--archive-sha256", ARCHIVE_SHA256,
    "--out", "/content/parity-gpu-out",
]
if RUN == "p0005":
    command += ["--only", "p0005"]
elif RUN != "full":
    raise RuntimeError(f'RUN must be "full" or "p0005", got {RUN!r}. Fix cell 2 and run it again.')
RESULT_ZIP = None
proc = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
    if line.startswith("MODEL_ZIP="):  # one model finished: download its results now, so a lost session keeps them
        files.download(line.split("=", 1)[1].strip())
    if line.startswith("RESULT_ZIP="):
        RESULT_ZIP = line.split("=", 1)[1].strip()
exit_code = proc.wait()
print(f"\nExit code {exit_code}. Cell 6 is optional; cell 7 downloads the results zip, whether or not this succeeded.")

In [ ]:
# Cell 6 (optional): the P-0005 repro on this GPU, a few minutes. You can skip this cell.
# Its output goes into the same results zip, in its own P-0005 folder, apart from the verdicts.
# It never changes the main results: if it fails, the zip from cell 5 stays exactly as it was.
import subprocess
import sys
if not globals().get("RESULT_ZIP"):
    raise RuntimeError("Run cell 5 first: this cell adds to the results zip that cell 5 writes.")
if RUN == "p0005":
    raise RuntimeError('Skip this cell: with RUN = "p0005", cell 5 already ran P-0005. Go to cell 7.')
command = [
    sys.executable, str(ROOT / "docs" / "gpu-run" / "colab_kit.py"),
    "--extra", "p0005", "--out", "/content/parity-gpu-out", "--zip", RESULT_ZIP,
]
proc = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
p0005_exit = proc.wait()
print(f"\nP-0005 exit code {p0005_exit}. Either way, run cell 7 to download the results zip.")


In [ ]:
# Cell 7: download the results zip.
from google.colab import files
if not RESULT_ZIP:
    raise RuntimeError("No results zip was written. Save the whole output of cell 5 as a text file and hand that back.")
print(f"Downloading {RESULT_ZIP}")
files.download(RESULT_ZIP)